In [ ]:
from os import listdir
from experiment_main import *


def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    )


def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # (rule_data[i][11]), # / ticker_length)
                # rule_data[i][12],
                # (rule_data[i][13] * 100 / rule_data[i][12]) if rule_data[i][12] != 0 else 0,
                # (rule_data[i][14]), # / ticker_length)
                # rule_data[i][15],
                # (rule_data[i][16] * 100 / rule_data[i][15]) if rule_data[i][15] != 0 else 0,
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]


def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            # "Long 1 Return",
            # "Long 1 Transactions",
            # "Long 1 Win Ratio",
            # "Long 2 Return",
            # "Long 2 Transactions",
            # "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)

In [2]:
market = "soybean"
time_format = "1_day"

macd_params_basic = (12, 26, 9)
macd_params = macd_params_basic

# macd_heuristics_params = (11.017, 32.925, 16.467)
# macd_heuristics_params = (11, 33, 16)
# macd_params = macd_heuristics_params

# macd_crossover_heuristics_params = (6.866, 33.812, 4.575)
# macd_crossover_heuristics_params = (7, 34, 5)
# macd_params = macd_crossover_heuristics_params

params_str = "_".join(str(val) for val in macd_params)
short_period, long_period, signal_period = macd_params

trade_cost = False
balance = 0
risk_per_trade = 100 / 100
last_close = True

prefix = f"./data/{market}/"

soybean_data = pd.read_csv(
    prefix + "soy_data.csv",
    index_col="Date",
    parse_dates=True,
    sep=";",
    usecols=["Date", "Close", "Open", "High", "Low"],
)

soybean_data.dropna(inplace=True)
soybean_data.columns = list(map(lambda x: x.lower(), list(soybean_data.columns)))
soybean_data.sort_index(inplace=True)
soybean_data.index.names = ["date"]

soybean_data = calculate_indicators(
    soybean_data,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
)

soybean_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2011-02-18,1400.50,1405.25,1373.50,1375.25,-0.777474,9.477944,-10.255418,-3.933921,8.431267,-12.365189,...,-2.870952,966,12.209431,15.080383,-2.870952,977,12.209431,15.080383,-2.870952,982
2011-02-22,1344.75,1361.75,1311.00,1311.00,-7.908077,6.000740,-13.908817,-11.484817,4.448050,-15.932868,...,-3.335714,966,10.910741,14.246454,-3.335714,977,10.910741,14.246454,-3.335714,982
2011-02-23,1317.50,1333.00,1314.50,1333.00,-11.649626,2.470667,-14.120292,-11.797953,1.198850,-12.996803,...,-4.174076,966,9.028859,13.202935,-4.174076,977,9.028859,13.202935,-4.174076,982
2011-02-24,1324.25,1334.75,1311.00,1328.50,-14.807253,-0.984917,-13.822336,-17.515624,-2.544045,-14.971579,...,-4.993540,966,6.961010,11.954550,-4.993540,977,6.961010,11.954550,-4.993540,982
2011-02-25,1339.75,1398.50,1338.25,1371.75,-13.662292,-3.520392,-10.141900,-10.194294,-4.074095,-6.120199,...,-5.736293,966,4.784184,10.520477,-5.736293,977,4.784184,10.520477,-5.736293,982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-24,1036.12,1041.38,1025.62,1028.62,3.563438,4.686156,-1.122718,2.797186,4.486147,-1.688961,...,-0.834284,966,6.009692,6.843977,-0.834285,977,6.009692,6.843977,-0.834285,982
2014-12-26,1032.12,1047.88,1030.88,1047.12,4.452187,4.639362,-0.187175,5.941688,4.777255,1.164433,...,-0.863128,966,5.765066,6.628195,-0.863128,977,5.765066,6.628195,-0.863128,982
2014-12-29,1053.50,1067.75,1046.12,1049.62,5.297193,4.770928,0.526265,4.717148,4.765234,-0.048086,...,-0.795533,966,5.633778,6.429311,-0.795533,977,5.633778,6.429311,-0.795533,982
2014-12-30,1048.62,1055.88,1042.38,1042.62,5.340464,4.884835,0.455628,5.901448,4.992477,0.908971,...,-0.663353,966,5.600120,6.263473,-0.663353,977,5.600120,6.263473,-0.663353,982


In [3]:
start_optimize_soybean = pd.Timestamp("2012-01-03")
end_optimize_soybean = pd.Timestamp("2013-12-24")
start_test_soybean = pd.Timestamp("2013-12-26")
end_test_soybean = pd.Timestamp("2014-08-14")

optimize_data = soybean_data[start_optimize_soybean:end_optimize_soybean]
test_data = soybean_data[start_test_soybean:end_test_soybean]

# optimize_data
test_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2013-12-26,1329.50,1329.50,1316.75,1319.75,6.127142,8.510769,-2.383627,5.695191,8.212649,-2.517458,...,0.778431,966,8.133636,7.355205,0.778431,977,8.133636,7.355205,0.778431,982
2013-12-27,1318.75,1332.38,1318.75,1331.75,5.862936,7.981202,-2.118266,6.013907,7.772900,-1.758994,...,0.462226,966,7.932987,7.470761,0.462226,977,7.932987,7.470761,0.462226,982
2013-12-30,1313.60,1315.88,1299.88,1308.88,3.764735,7.137909,-3.373173,1.740481,6.566417,-4.825935,...,0.204176,966,7.725982,7.521805,0.204176,977,7.725982,7.521805,0.204176,982
2013-12-31,1307.38,1310.88,1289.38,1289.38,0.522388,5.814805,-5.292417,-0.573897,5.138354,-5.712251,...,-0.153559,966,7.329857,7.483416,-0.153559,977,7.329857,7.483416,-0.153559,982
2014-01-02,1288.88,1288.88,1262.88,1270.75,-3.510020,3.949840,-7.459860,-6.152532,2.880177,-9.032708,...,-0.667444,966,6.649110,7.316555,-0.667444,977,6.649110,7.316555,-0.667444,982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-08-08,1098.00,1114.12,1097.38,1113.88,-54.800078,-58.797472,3.997394,-49.743201,-57.838784,8.095584,...,-2.686904,966,-56.438191,-53.751287,-2.686904,977,-56.438191,-53.751287,-2.686904,982
2014-08-11,1111.75,1128.62,1105.88,1108.00,-51.766811,-57.391340,5.624529,-53.588060,-56.988639,3.400579,...,-2.018474,966,-56.274380,-54.255906,-2.018474,977,-56.274380,-54.255906,-2.018474,982
2014-08-12,1109.00,1114.25,1079.88,1094.75,-49.857365,-55.884545,6.027180,-46.499739,-54.890859,8.391121,...,-1.254174,966,-55.823623,-54.569449,-1.254174,977,-55.823623,-54.569449,-1.254174,982
2014-08-13,1093.50,1102.62,1078.88,1079.50,-49.009709,-54.509577,5.499869,-51.268682,-54.166424,2.897742,...,-0.526038,966,-55.226997,-54.700959,-0.526038,977,-55.226997,-54.700959,-0.526038,982


SYSTEM 1 - NO EXTRA CONDITION

In [4]:
signal_line_if = False
signal_line_if_target = False

Signalline crossover

In [ ]:
sys1_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys1_major_macd_crossover)):
    sys1_major_macd_crossover[i].extend(sys1_major_macd_crossover[i][-3:])
    print(sys1_major_macd_crossover[i], len(sys1_major_macd_crossover[i]))

best_sys1_major_macd_crossover = get_best_results(sys1_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[43.719999999999345, 3.643333333333279, 0, 12, 4, 6, 2, -88.77000000000044, 6, 2, 132.48999999999978, -88.77000000000044, 6, 2, -88.77000000000044, 6, 2] 17
[94.76000000000022, 2.961250000000007, 0, 32, 14, 16, 7, -54.61999999999989, 16, 7, 149.3800000000001, -54.61999999999989, 16, 7, -54.61999999999989, 16, 7] 17
[94.76000000000022, 2.961250000000007, 0, 32, 14, 16, 7, -54.61999999999989, 16, 7, 149.3800000000001, -54.61999999999989, 16, 7, -54.61999999999989, 16, 7] 17
[-85.02000000000044, -7.085000000000036, 0, 12, 2, 6, 1, -150.76000000000022, 6, 1, 65.73999999999978, -150.76000000000022, 6, 1, -150.76000000000022, 6, 1] 17
[41.45999999999913, 3.4549999999999272, 0, 12, 4, 6, 2, -88.77000000000044, 6, 2, 130.22999999999956, -88.77000000000044, 6, 2, -88.77000000000044, 6, 2] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
3,43.72,3.643333,0,12,4,6,2,-88.77,6,2,132.49,-88.77,6,2,Histogram a: -0.6
0,94.76,2.961250,0,32,14,16,7,-54.62,16,7,149.38,-54.62,16,7,Histogram a: -0.9
0,94.76,2.961250,0,32,14,16,7,-54.62,16,7,149.38,-54.62,16,7,Histogram a: -0.9
147,-85.02,-7.085000,0,12,2,6,1,-150.76,6,1,65.74,-150.76,6,1,Histogram a: -0.4 g: 60
9,41.46,3.455000,0,12,4,6,2,-88.77,6,2,130.23,-88.77,6,2,Histogram a: 0


Zeroline crossover

In [ ]:
sys1_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys1_major_zero_crossover)):
    sys1_major_zero_crossover[i].extend(sys1_major_zero_crossover[i][-3:])
    print(sys1_major_zero_crossover[i], len(sys1_major_zero_crossover[i]))

best_sys1_major_zero_crossover = get_best_results(sys1_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[82.73999999999978, 41.36999999999989, 0, 2, 1, 1, 1, 136.3699999999999, 1, 0, -53.63000000000011, 136.3699999999999, 1, 1, 136.3699999999999, 1, 1] 17
[11.61999999999989, 5.809999999999945, 0, 2, 1, 1, 1, 79.86999999999989, 1, 0, -68.25, 79.86999999999989, 1, 1, 79.86999999999989, 1, 1] 17
[55.14000000000033, 27.570000000000164, 0, 2, 1, 1, 1, 103.26000000000022, 1, 0, -48.11999999999989, 103.26000000000022, 1, 1, 103.26000000000022, 1, 1] 17
[55.14000000000033, 27.570000000000164, 0, 2, 1, 1, 1, 103.26000000000022, 1, 0, -48.11999999999989, 103.26000000000022, 1, 1, 103.26000000000022, 1, 1] 17
[76.36999999999989, 38.184999999999945, 0, 2, 1, 1, 1, 131.6199999999999, 1, 0, -55.25, 131.6199999999999, 1, 1, 131.6199999999999, 1, 1] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
357,82.74,41.370,0,2,1,1,1,136.37,1,0,-53.63,136.37,1,1,MACD a: 0.5 g: 4
18,11.62,5.810,0,2,1,1,1,79.87,1,0,-68.25,79.87,1,1,MACD a: 0.9
427,55.14,27.570,0,2,1,1,1,103.26,1,0,-48.12,103.26,1,1,MACD a: 0.8 g: 1
444,55.14,27.570,0,2,1,1,1,103.26,1,0,-48.12,103.26,1,1,MACD a: 0.8 g: 600
9,76.37,38.185,0,2,1,1,1,131.62,1,0,-55.25,131.62,1,1,MACD a: 0


SYSTEM 2 - SIGNAL EXTRA CONDITION

In [7]:
signal_line_if = True
signal_line_if_target = False

Signalline crossover

In [ ]:
sys2_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys2_major_macd_crossover)):
    sys2_major_macd_crossover[i].extend(sys2_major_macd_crossover[i][-3:])
    print(sys2_major_macd_crossover[i], len(sys2_major_macd_crossover[i]))

best_sys2_major_macd_crossover = get_best_results(sys2_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[259.7600000000002, 129.8800000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 181.76000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[263.2600000000002, 131.6300000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 185.26000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[263.2600000000002, 131.6300000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 185.26000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[259.7600000000002, 129.8800000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 181.76000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[256.8699999999999, 128.43499999999995, 0, 2, 2, 1, 1, 69.86999999999989, 1, 1, 187.0, 69.86999999999989, 1, 1, 69.86999999999989, 1, 1] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
307,259.76,129.880,0,2,2,1,1,78.00,1,1,181.76,78.00,1,1,Histogram a: 0.3 g: 1
14,263.26,131.630,0,2,2,1,1,78.00,1,1,185.26,78.00,1,1,Histogram a: 0.5
213,263.26,131.630,0,2,2,1,1,78.00,1,1,185.26,78.00,1,1,Histogram a: -0.1 g: 4
283,259.76,129.880,0,2,2,1,1,78.00,1,1,181.76,78.00,1,1,Histogram a: 0.2 g: 1
9,256.87,128.435,0,2,2,1,1,69.87,1,1,187.00,69.87,1,1,Histogram a: 0


Zeroline crossover

In [ ]:
sys2_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys2_major_zero_crossover)):
    sys2_major_zero_crossover[i].extend(sys2_major_zero_crossover[i][-3:])
    print(sys2_major_zero_crossover[i], len(sys2_major_zero_crossover[i]))

best_sys2_major_zero_crossover = get_best_results(sys2_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[65.76000000000022, 32.88000000000011, 0, 2, 1, 1, 1, 125.13000000000011, 1, 0, -59.36999999999989, 125.13000000000011, 1, 1, 125.13000000000011, 1, 1] 17
[115.23999999999978, 57.61999999999989, 0, 2, 1, 1, 1, 141.73999999999978, 1, 0, -26.5, 141.73999999999978, 1, 1, 141.73999999999978, 1, 1] 17
[71.26000000000022, 35.63000000000011, 0, 2, 1, 1, 1, 130.6300000000001, 1, 0, -59.36999999999989, 130.6300000000001, 1, 1, 130.6300000000001, 1, 1] 17
[71.26000000000022, 35.63000000000011, 0, 2, 1, 1, 1, 130.6300000000001, 1, 0, -59.36999999999989, 130.6300000000001, 1, 1, 130.6300000000001, 1, 1] 17
[76.36999999999989, 38.184999999999945, 0, 2, 1, 1, 1, 131.6199999999999, 1, 0, -55.25, 131.6199999999999, 1, 1, 131.6199999999999, 1, 1] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
215,65.76,32.880,0,2,1,1,1,125.13,1,0,-59.37,125.13,1,1,MACD a: -0.1 g: 20
0,115.24,57.620,0,2,1,1,1,141.74,1,0,-26.50,141.74,1,1,MACD a: -0.9
283,71.26,35.630,0,2,1,1,1,130.63,1,0,-59.37,130.63,1,1,MACD a: 0.2 g: 1
319,71.26,35.630,0,2,1,1,1,130.63,1,0,-59.37,130.63,1,1,MACD a: 0.3 g: 100
9,76.37,38.185,0,2,1,1,1,131.62,1,0,-55.25,131.62,1,1,MACD a: 0


SYSTEM 3 - Target SIGNAL EXTRA CONDITION

In [10]:
signal_line_if = False
signal_line_if_target = True

Signalline crossover

In [ ]:
sys3_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys3_major_macd_crossover)):
    sys3_major_macd_crossover[i].extend(sys3_major_macd_crossover[i][-3:])
    print(sys3_major_macd_crossover[i], len(sys3_major_macd_crossover[i]))

best_sys3_major_macd_crossover = get_best_results(sys3_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[259.7600000000002, 129.8800000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 181.76000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[263.2600000000002, 131.6300000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 185.26000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[263.2600000000002, 131.6300000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 185.26000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[259.7600000000002, 129.8800000000001, 0, 2, 2, 1, 1, 78.0, 1, 1, 181.76000000000022, 78.0, 1, 1, 78.0, 1, 1] 17
[256.8699999999999, 128.43499999999995, 0, 2, 2, 1, 1, 69.86999999999989, 1, 1, 187.0, 69.86999999999989, 1, 1, 69.86999999999989, 1, 1] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
307,259.76,129.880,0,2,2,1,1,78.00,1,1,181.76,78.00,1,1,Histogram a: 0.3 g: 1
14,263.26,131.630,0,2,2,1,1,78.00,1,1,185.26,78.00,1,1,Histogram a: 0.5
213,263.26,131.630,0,2,2,1,1,78.00,1,1,185.26,78.00,1,1,Histogram a: -0.1 g: 4
283,259.76,129.880,0,2,2,1,1,78.00,1,1,181.76,78.00,1,1,Histogram a: 0.2 g: 1
9,256.87,128.435,0,2,2,1,1,69.87,1,1,187.00,69.87,1,1,Histogram a: 0


Zerocross crossover

In [ ]:
sys3_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys3_major_zero_crossover)):
    sys3_major_zero_crossover[i].extend(sys3_major_zero_crossover[i][-3:])
    print(sys3_major_zero_crossover[i], len(sys3_major_zero_crossover[i]))

best_sys3_major_zero_crossover = get_best_results(sys3_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[65.76000000000022, 32.88000000000011, 0, 2, 1, 1, 1, 125.13000000000011, 1, 0, -59.36999999999989, 125.13000000000011, 1, 1, 125.13000000000011, 1, 1] 17
[11.61999999999989, 5.809999999999945, 0, 2, 1, 1, 1, 79.86999999999989, 1, 0, -68.25, 79.86999999999989, 1, 1, 79.86999999999989, 1, 1] 17
[71.26000000000022, 35.63000000000011, 0, 2, 1, 1, 1, 130.6300000000001, 1, 0, -59.36999999999989, 130.6300000000001, 1, 1, 130.6300000000001, 1, 1] 17
[71.26000000000022, 35.63000000000011, 0, 2, 1, 1, 1, 130.6300000000001, 1, 0, -59.36999999999989, 130.6300000000001, 1, 1, 130.6300000000001, 1, 1] 17
[76.36999999999989, 38.184999999999945, 0, 2, 1, 1, 1, 131.6199999999999, 1, 0, -55.25, 131.6199999999999, 1, 1, 131.6199999999999, 1, 1] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
215,65.76,32.880,0,2,1,1,1,125.13,1,0,-59.37,125.13,1,1,MACD a: -0.1 g: 20
18,11.62,5.810,0,2,1,1,1,79.87,1,0,-68.25,79.87,1,1,MACD a: 0.9
283,71.26,35.630,0,2,1,1,1,130.63,1,0,-59.37,130.63,1,1,MACD a: 0.2 g: 1
305,71.26,35.630,0,2,1,1,1,130.63,1,0,-59.37,130.63,1,1,MACD a: 0.2 g: 10000
9,76.37,38.185,0,2,1,1,1,131.62,1,0,-55.25,131.62,1,1,MACD a: 0


Getting Tables Detailed

Signalline crossover

In [13]:
sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:

        if col[-6:] == "Return":
            l.append(round(100 * x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/soybean/soy_rules_macdcross_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Rule 1   4372.0            12      33.33      -8877.0   
         Rule 2   9476.0            32      43.75      -5462.0   
         Rule 3   9476.0            32      43.75      -5462.0   
         GA      -8502.0            12      16.67     -15076.0   
         MACD     4146.0            12      33.33      -8877.0   
System 2 Rule 1  25976.0             2     100.00       7800.0   
         Rule 2  26326.0             2     100.00       7800.0   
         Rule 3  26326.0             2     100.00       7800.0   
         GA      25976.0             2     100.00       7800.0   
         MACD    25687.0             2     100.00       6987.0   
System 3 Rule 1  25976.0             2     100.00       7800.0   
         Rule 2  26326.0             2     100.00       7800.0   
         Rule 3  26326.0             2     100.00       7800.0   
         GA      25976.0             2     100.00       7800.0   
         MACD    25687.0             2     100.00       6987.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                  6           33.33       13249.0   
         Rule 2                 16           43.75       14938.0   
         Rule 3                 16           43.75       14938.0   
         GA                      6           16.67        6574.0   
         MACD                    6           33.33       13023.0   
System 2 Rule 1                  1          100.00       18176.0   
         Rule 2                  1          100.00       18526.0   
         Rule 3                  1          100.00       18526.0   
         GA                      1          100.00       18176.0   
         MACD                    1          100.00       18700.0   
System 3 Rule 1                  1          100.00       18176.0   
         Rule 2                  1          100.00       18526.0   
         Rule 3                  1          100.00       18526.0   
         GA                      1          100.00       18176.0   
         MACD                    1          100.00       18700.0   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            33.33  
         Rule 2            43.75  
         Rule 3            43.75  
         GA                16.67  
         MACD              33.33  
System 2 Rule 1           100.00  
         Rule 2           100.00  
         Rule 3           100.00  
         GA               100.00  
         MACD             100.00  
System 3 Rule 1           100.00  
         Rule 2           100.00  
         Rule 3           100.00  
         GA               100.00  
         MACD             100.00

In [14]:
# sys_modes = ["System 1", "System 2", "System 3"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
data = [
    [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
    [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        if col[-6:] == "Return":
            l.append(round(100 * x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/soybean/soy_rules_macdcross_overall_{params_str}.csv", index=True
)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Best     9476.0            32      43.75      -5462.0   
         MACD     4146.0            12      33.33      -8877.0   
System 2 Best    26326.0             2     100.00       7800.0   
         MACD    25687.0             2     100.00       6987.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Best                   16           43.75       14938.0   
         MACD                    6           33.33       13023.0   
System 2 Best                    1          100.00       18526.0   
         MACD                    1          100.00       18700.0   

                 Short Win Ratio  
first    second                   
System 1 Best              43.75  
         MACD              33.33  
System 2 Best             100.00  
         MACD             100.00

Zeroline crossover

In [15]:
sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# data = [sys1_major_zero_crossover, sys2_major_zero_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        if col[-6:] == "Return":
            l.append(round(100 * x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/soybean/soy_rules_zerocross_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Rule 1   8274.0             2       50.0      13637.0   
         Rule 2   1162.0             2       50.0       7987.0   
         Rule 3   5514.0             2       50.0      10326.0   
         GA       5514.0             2       50.0      10326.0   
         MACD     7637.0             2       50.0      13162.0   
System 2 Rule 1   6576.0             2       50.0      12513.0   
         Rule 2  11524.0             2       50.0      14174.0   
         Rule 3   7126.0             2       50.0      13063.0   
         GA       7126.0             2       50.0      13063.0   
         MACD     7637.0             2       50.0      13162.0   
System 3 Rule 1   6576.0             2       50.0      12513.0   
         Rule 2   1162.0             2       50.0       7987.0   
         Rule 3   7126.0             2       50.0      13063.0   
         GA       7126.0             2       50.0      13063.0   
         MACD     7637.0             2       50.0      13162.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                  1           100.0       -5363.0   
         Rule 2                  1           100.0       -6825.0   
         Rule 3                  1           100.0       -4812.0   
         GA                      1           100.0       -4812.0   
         MACD                    1           100.0       -5525.0   
System 2 Rule 1                  1           100.0       -5937.0   
         Rule 2                  1           100.0       -2650.0   
         Rule 3                  1           100.0       -5937.0   
         GA                      1           100.0       -5937.0   
         MACD                    1           100.0       -5525.0   
System 3 Rule 1                  1           100.0       -5937.0   
         Rule 2                  1           100.0       -6825.0   
         Rule 3                  1           100.0       -5937.0   
         GA                      1           100.0       -5937.0   
         MACD                    1           100.0       -5525.0   

                 Short Win Ratio  
first    second                   
System 1 Rule 1              0.0  
         Rule 2              0.0  
         Rule 3              0.0  
         GA                  0.0  
         MACD                0.0  
System 2 Rule 1              0.0  
         Rule 2              0.0  
         Rule 3              0.0  
         GA                  0.0  
         MACD                0.0  
System 3 Rule 1              0.0  
         Rule 2              0.0  
         Rule 3              0.0  
         GA                  0.0  
         MACD                0.0

In [16]:
# sys_modes = ["System 1", "System 2", "System 3"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]], [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]], [best_sys3_major_zero_crossover, sys3_major_zero_crossover[-1]]]
data = [
    [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
    [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        if col[-6:] == "Return":
            l.append(round(100 * x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/soybean/soy_rules_zerocross_overall_{params_str}.csv", index=True
)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Best     8274.0             2       50.0      13637.0   
         MACD     7637.0             2       50.0      13162.0   
System 2 Best    11524.0             2       50.0      14174.0   
         MACD     7637.0             2       50.0      13162.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Best                    1           100.0       -5363.0   
         MACD                    1           100.0       -5525.0   
System 2 Best                    1           100.0       -2650.0   
         MACD                    1           100.0       -5525.0   

                 Short Win Ratio  
first    second                   
System 1 Best                0.0  
         MACD                0.0  
System 2 Best                0.0  
         MACD                0.0